# Run 2 Validation

This notebook validates the output tables after Run 2 (Day 2) data load.

## Expected Changes in Run 2:
- **Customer Updates**:
  - John (ID=1): Email changed from john.doe@example.com to jdoe@example.com
  - Richard (ID=10): Marked for deletion (DELETE_FLAG=1)
  - New customers: Alice (ID=3), Joe (ID=4)
- **Customer Address Updates**:
  - Jane (ID=2): City changed to Perth, WA
  - New: Alice (ID=3) in Sydney, NSW
- **SCD2 Behavior**: Previous versions should be closed, new versions created


In [ ]:
%run "./initialize"


In [ ]:
# Import validation utilities
from validation_utils import ValidationRunner

# Initialize validation runner with spark session
v = ValidationRunner(spark)


## Bronze Layer Validations

After Run 2:
- `customer`: 7 rows total (3 original + 4 new CDC records)
- `customer_address`: 6 rows (4 original + 2 new)


In [ ]:
print("=" * 60)
print("BRONZE LAYER - Base Samples")
print("=" * 60)

# Bronze customer - 3 original + 4 new (John update, Alice, Joe, Richard delete marker)
v.validate_row_count(f"{bronze_schema}.customer", 7, "Total customer CDC records")

# All customer IDs including new ones
v.validate_values_exist(f"{bronze_schema}.customer", "CUSTOMER_ID", [1, 2, 3, 4, 10], "All customer IDs")

# Bronze customer_address - 4 original + 2 new (Jane update, Alice new)
v.validate_row_count(f"{bronze_schema}.customer_address", 6, "Total address CDC records")


## Silver Layer Validations

### SCD2 Behavior After Run 2:
- **customer**: 
  - John (ID=1): 1 closed (old email), 1 active (new email)
  - Jane (ID=2): 1 active (unchanged)
  - Alice (ID=3): 1 active (new)
  - Joe (ID=4): 1 active (new)
  - Richard (ID=10): 1 closed (deleted via apply_as_deletes)
  - Total: 4 active, 2 closed = 6 rows


In [ ]:
print("\n" + "=" * 60)
print("SILVER LAYER - Base Samples")
print("=" * 60)

# Silver customer SCD2 after Day 2:
# Active: Jane (unchanged), Alice (new), Joe (new), John (updated) = 4
# Closed: John (old version), Richard (deleted) = 2
v.validate_active_scd2_count(f"{silver_schema}.customer", 4, "__END_AT")
v.validate_closed_scd2_count(f"{silver_schema}.customer", 2, "__END_AT")

# Validate John's current email is updated
v.validate_column_value(
    f"{silver_schema}.customer",
    "CUSTOMER_ID = 1 AND __END_AT IS NULL",
    "EMAIL",
    "jdoe@example.com",
    "John's updated email (active record)"
)

# Validate new customers exist
v.validate_values_exist(f"{silver_schema}.customer", "CUSTOMER_ID", [1, 2, 3, 4], "Active customer IDs")

# Silver customer_address SCD2:
# Jane (ID=2): 1 closed (Melbourne), 1 active (Perth)
# Alice (ID=3): 1 active (new)
# Original 1, 4, 10 still active
v.validate_min_closed_scd2_count(f"{silver_schema}.customer_address", 1, "__END_AT")


### Multi-Source Streaming Pipeline

The multi-source streaming table merges customer and customer_address data.


In [ ]:
print("\n" + "=" * 60)
print("SILVER LAYER - Multi-Source Streaming")
print("=" * 60)

# After Run 2, multi-source should have:
# Active IDs: 1, 2, 3, 4 (10 was deleted)
# Should have historical records from changes
v.validate_active_scd2_count(
    f"{silver_schema}.customer_ms_basic",
    4,
    "__END_AT"
)

# Should have at least 1 closed record from changes
v.validate_min_closed_scd2_count(
    f"{silver_schema}.customer_ms_basic",
    1,
    "__END_AT"
)

print("\n" + "=" * 60)
print("OPERATIONAL METADATA - meta_load_details Validation")
print("=" * 60)

# Validate meta_load_details nested column fields are not null
# Using wildcard to check all fields in the struct
v.validate_column_not_null(
    f"{silver_schema}.customer",
    "meta_load_details.pipeline_start_timestamp",
    "Silver customer meta_load_details"
)

v.validate_column_not_null(
    f"{silver_schema}.customer_address",
    "meta_load_details.pipeline_start_timestamp",
    "Silver customer_address meta_load_details"
)

v.validate_column_not_null(
    f"{silver_schema}.customer_ms_basic",
    "meta_load_details.pipeline_start_timestamp",
    "Silver customer_ms_basic meta_load_details"
)


## Gold Layer Validations


In [ ]:
print("\n" + "=" * 60)
print("GOLD LAYER - Stream-Static")
print("=" * 60)

# Gold dimension should reflect silver layer changes
v.validate_min_row_count(
    f"{gold_schema}.dim_customer_sql_sample",
    4,  # At least 4 rows (active records)
    "Dimension records"
)


## Feature Samples - Snapshots


In [ ]:
print("\n" + "=" * 60)
print("BRONZE LAYER - Feature Samples (Snapshots)")
print("=" * 60)

# Periodic snapshot should show SCD2 changes after snapshot update
# Run 2 overwrites snapshot source, so periodic snapshot should detect changes
v.validate_min_row_count(
    f"{bronze_schema}.feature_periodic_snapshot_scd2",
    3,  # At least 3 records from initial + any changes
    "Periodic snapshot records"
)


## Validation Summary


In [ ]:
# Validation summary runs at the end of the notebook after all checks complete
pass

In [ ]:
print("\n" + "=" * 60)
print("BRONZE LAYER - Feature Snapshots (Additional Tables) Run 2")
print("=" * 60)

# Periodic snapshot SCD1: customer_snapshot_source OVERWRITTEN in Run 2 to 5 rows
# (John jdoe@, Jane, Alice, Joe, Richard) - SCD1 full replacement
v.validate_row_count(
    f"{bronze_schema}.feature_periodic_snapshot_scd1",
    5,
    "Periodic snapshot SCD1 (5 customers after snapshot overwrite)"
)

# Historical file snapshots (datetime): new snapshot file 2024-03-01 adds customer 6 (Someone)
# Active now includes customer 6; at least 7 rows total (was 6, now +1 active)
v.validate_min_row_count(
    f"{bronze_schema}.feature_historical_snapshot_files_datetime",
    7,
    "Historical snapshot files (datetime) - at least 7 rows after 4th snapshot"
)

# Integer-versioned historical snapshot: customer_2.csv now loaded (John jdoe@, Alice, Joe)
# SCD2: John gets new version (email changed), Alice/Joe added, customer_1.csv history preserved
# Active: John (jdoe@), Alice, Joe = 3; Closed: John (john.doe@), Jane = 2; Total >= 5
v.validate_min_row_count(
    f"{bronze_schema}.feature_historical_snapshot_files_int",
    4,
    "Historical snapshot int (at least 4 rows after customer_2.csv loaded)"
)

print("\n" + "=" * 60)
print("BRONZE LAYER - Feature Table Migration Pipeline Run 2")
print("=" * 60)

# feature_migrated_table_scd2: Run 2 adds streaming CDC records (John update, Alice, Joe)
# Active: 5 records after Run 2 streaming
# Closed: at least 1 (from SCD2 version history)
v.validate_active_scd2_count(
    f"{bronze_schema}.feature_migrated_table_scd2",
    5,
    "__END_AT"
)
v.validate_min_closed_scd2_count(
    f"{bronze_schema}.feature_migrated_table_scd2",
    1,
    "__END_AT"
)

# feature_migrated_table_append_only: rows accumulated from streaming after Run 2
v.validate_min_row_count(
    f"{bronze_schema}.feature_migrated_table_append_only",
    3,
    "Append-only migration (at least 3 rows after Run 2 streaming)"
)

print("\n" + "=" * 60)
print("BRONZE LAYER - Template Samples After Run 2")
print("=" * 60)

# customer_template: Run 2 adds 2024-03-01 snapshot file with customer 6
# Active: 6 records after Run 2 snapshot ingestion
v.validate_active_scd2_count(
    f"{bronze_schema}.customer_template",
    6,
    "__END_AT"
)

# customer_address_template: Run 2 adds 2024-03-01 file (Jane Perth, Alice Sydney, Someone Adelaide)
# ID 2 (Jane) moves to Perth → new version; new IDs 3 (Sydney), 6 (Adelaide)
v.validate_min_row_count(
    f"{bronze_schema}.customer_address_template",
    5,
    "Template customer_address SCD2 (at least 5 rows after Run 2 snapshot)"
)

## Feature Snapshots Pipeline - Additional Tables (Run 2)

- `feature_periodic_snapshot_scd1`: customer_snapshot_source OVERWRITTEN to 5 rows (John, Jane, Alice, Joe, Richard)
- `feature_historical_snapshot_files_datetime`: new snapshot file 2024-03-01 (customer 6) adds a row, total at least 7
- `feature_historical_snapshot_files_int`: customer_2.csv now loaded: John (jdoe@), Alice, Joe → at least 4 rows (history + new)
- `feature_migrated_table_scd2`: John and Richard get new versions; Alice and Joe added

In [ ]:
print("\n" + "=" * 60)
print("BRONZE LAYER - Feature General Pipeline")
print("=" * 60)

# append_sql_flow: 5 from Run 1 + 2 new (Jane Perth, Alice Sydney) = 7 total
v.validate_row_count(f"{bronze_schema}.append_sql_flow", 7, "Address rows after Run 2 (+2 new)")

# append_view_flow: 3 from Run 1 + 4 new customer CDC records = 7 total
v.validate_row_count(f"{bronze_schema}.append_view_flow", 7, "Customer rows after Run 2 (+4 new CDC)")

# feature_constraints: 3 original + 4 new = 7 total rows (streaming append from customer CDC)
v.validate_row_count(f"{bronze_schema}.feature_constraints", 7, "Customer rows with DDL constraints after Run 2")

# feature_version_mapping_flows: SCD1, updated in-place
# John updated to jdoe@, Alice + Joe inserted, Richard updated (DELETE_FLAG=True)
v.validate_row_count(f"{bronze_schema}.feature_version_mapping_flows", 5, "SCD1 version mapping (5 customers after Run 2)")
v.validate_column_value(
    f"{bronze_schema}.feature_version_mapping_flows",
    "CUSTOMER_ID = 1",
    "EMAIL",
    "jdoe@example.com",
    "John's updated email in SCD1 table"
)

# v_version_mapping_standard: same SCD1 behaviour
v.validate_row_count(f"{bronze_schema}.v_version_mapping_standard", 5, "SCD1 standard (5 customers after Run 2)")

print("\n" + "=" * 60)
print("BRONZE LAYER - Materialized Views After Run 2")
print("=" * 60)

# MVs source from staging.customer (7 rows after Run 2: 3 original + 4 new CDC)
v.validate_row_count(f"{bronze_schema}.feature_mv_source_view", 7, "MV source view (7 rows after Run 2)")
v.validate_row_count(f"{bronze_schema}.feature_mv_sql_path", 7, "MV SQL path (7 rows after Run 2)")
v.validate_row_count(f"{bronze_schema}.feature_mv_chain_mvs", 7, "Chained MV (7 rows after Run 2)")

# feature_mv_with_quarantine: sources from staging.customer_address
# Run 1: 5 rows; Run 2: +2 (Jane Perth, Alice Sydney) = 7 total, 1 NULL quarantined → 6 valid
v.validate_row_count(f"{bronze_schema}.feature_mv_with_quarantine", 6, "MV with quarantine (6 valid after Run 2)")

In [ ]:
v.print_summary()